# Data Understanding

## Import Libraries and Set Global Settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
%matplotlib inline

## Define Data Sources

### Load Datasets

In [ ]:
# Define Dataset Paths
datasets = {
    "uk_dataset": glob.glob("../data/raw/100,000 UK Used Car Data set/*.csv"),
    "andrei_dataset": "../data/raw/Used Cars Dataset Andrei Novikov/cars.csv"
}

# Assign dataframes variable
dfs = {}

# Iterate through datasets
for name, path in datasets.items():
    try:
        # Check if the path contains multiple files or is a single file
        if isinstance(path, list):
            # Read all files in the list, extract make, and concatenate them
            df_list = []
            for f in path:
                temp_df = pd.read_csv(f)
                
                # Extract filename without extension to use as the 'make'
                if name == "uk_dataset":
                    make_name = os.path.basename(f).replace('.csv', '').strip().lower()
                    
                    # Handle inconsistent make names from model named files (UK Dataset)
                    if make_name in ['merc', 'cclass']:
                        make_name = 'mercedes'
                    elif make_name == 'focus':
                        make_name = 'ford'
                        
                    # Add make names into the temporary dataframe
                    temp_df['make'] = make_name.capitalize()
                
                # Append the temporary dataframe with the updated make names to the dataframe list (UK Dataset)
                df_list.append(temp_df)
                
            # Append make names into a new feature in the dataset
            dfs[name] = pd.concat(df_list, ignore_index=True)
            
        else:
            # It's a single file path
            dfs[name] = pd.read_csv(path)
            
        # Output Dataset name and shape
        print(f"Loaded {name}: {dfs[name].shape[0]} rows, {dfs[name].shape[1]} columns.")
    except Exception as e:
        # Output Error if dataset not found
        print(f"Error loading {name}: {e}")

### Output Sample from All Datasets

In [ ]:
# Iterate through datasets
for i in dfs.keys():
    # Output Dataset Name
    print(f'\n{'='*20} {i} {'='*20}')
    # Output Samples of the dataset
    print(dfs[i].sample(5))

## Data Source Documentation

In [ ]:
# Document data source details for each dataset
data_source_report = {
    
    # 100,000 UK Dataset
    'uk_dataset': {
        'name': '100,000 UK Used Car Data set'
        'source': 'https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes',
        'acquisition_method': 'CSV download through kaggle.com (multiple files per car make)',
        'date_acquired': '2026',
        'issues_encountered': ['Multiple CSV files need merging', 'Inconsistent make names (cclass, merc, focus)']
    },
    
    # Andrei Novikov Dataset
    'andrei_dataset': {
        'name': 'Used Cars Dataset by Andrei Noviko'
        'source': 'https://www.kaggle.com/datasets/andreinovikov/used-cars-dataset',
        'acquisition_method': 'CSV download through kaggle.com',
        'date_acquired': '2026',
        'issues_encountered': ['Missing mpg and engineSize columns', 'US-based data (different market)']
    }
}

# Print the documented data to console/output
for name, report in data_source_report.items():
    print(f"\n{'='*20} {name} {'='*20}")
    for key, value in report.items():
        print(f"  {key}: {value}")

## Describe Data

### Dataset Information

In [ ]:
# Iterate through datasets
for name, df in dfs.items():
    # Output dataset name
    print(f"\n{'='*20} {name} {'='*20}")
    # Output dataset information
    print(df.info())

### Numerical Statistics

In [ ]:
# Iterate through datasets
for name, df in dfs.items():

    # Output dataset name header
    print(f"\n--- Numerical Statistics ({name}) ---")
    # Output dataset description
    print(df.describe()) 

### Categorical Statistics

In [ ]:
# Iterate through datasets
for name, df in dfs.items():

    # Categorical Overview
    if not df.select_dtypes(include='object').empty:

        # Output dataset name header
        print(f"\n--- Categorical Statistics ({name}) ---")
        # Output categorical feature Statistics
        print(df.describe(include='object'))

In [ ]:
# Iterate through datasets
for name, df in dfs.items():

    # Output dataset name header
    print(f"\n--- {name} Object Count ---")

    # Identify object-type columns in the current dataframe
    object_cols = df.select_dtypes(include='object').columns
    
    # Check if object columns exist before iterating
    if len(object_cols) > 0:
        # Iterate through identified categorical columns
        for col in object_cols:
            # Output column name header
            print(f"\n--- {col} ---")
            
            # Output top 20 value counts for the specific column
            print(df[col].value_counts().head(20)) 
    else:
        # Output message if no object columns are present in the dataset
        print(f"No object-type columns found in {name}.")

## Explore Data

### UK Dataset

In [ ]:
#### Price and Milage Distribution

In [ ]:
# Initialize subplots for price and mileage
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Set title for the combined plots
fig.suptitle('UK Dataset: Price and Mileage Distributions', fontsize=16)

# Generate histogram and KDE for price
sns.histplot(dfs['uk_dataset']['price'], bins=50, kde=True, ax=axes[0], color='skyblue')
    
# Set individual plot title and axis labels for price
axes[0].set_title('Distribution of Price')
axes[0].set_xlabel('Price (GBP)')

# Generate histogram and KDE for mileage
sns.histplot(dfs['uk_dataset']['mileage'], bins=50, kde=True, ax=axes[1], color='lightgreen')
    
# Set individual plot title and axis labels for mileage
axes[1].set_title('Distribution of Mileage')
axes[1].set_xlabel('Mileage')

# Adjust layout to prevent overlapping and display the plot
plt.tight_layout()
# Show histogram plots
plt.show()

#### Transmission and Fuel Type Count

In [ ]:
# Initialize subplots for transmission and fuel type
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Set title for the categorical analysis plots
fig.suptitle('UK Dataset: Transmission and Fuel Type Analysis', fontsize=16)

# Generate count plot for transmission types
sns.countplot(data=dfs['uk_dataset'], x='transmission', ax=axes[0], palette='Set2')
    
# Set individual plot title for transmission
axes[0].set_title('Count of Vehicles by Transmission')
    
# Annotate bars with frequency labels for better readability
for container in axes[0].containers:
    axes[0].bar_label(container, padding=3)

# Generate count plot for fuel types in the UK dataset
sns.countplot(data=dfs['uk_dataset'], x='fuelType', ax=axes[1], palette='Set2')
    
# Set individual plot title and apply log scale for better visualization of rare categories
axes[1].set_title('Count of Vehicles by Fuel Type (Log Scale)')
axes[1].set_yscale('log') 
axes[1].set_ylabel('Count (Log Scale)')
    
# Annotate bars with frequency labels (logarithmic values)
for container in axes[1].containers:
    axes[1].bar_label(container, padding=3)


# Adjust layout to optimize spacing between subplots
plt.tight_layout()
# Show countplots
plt.show()

#### Make and Model Distribution

In [ ]:
# Initialize subplots for make and model variety analysis
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Set title for the make-level overview
fig.suptitle('UK Dataset: Make and Model Analysis', fontsize=16)

# Determine the order of make by frequency for a cleaner visualization
make_order = dfs['uk_dataset']['make'].value_counts().index
    
# Generate horizontal count plot for make frequency
sns.countplot(data=dfs['uk_dataset'], y='make', order=make_order, ax=axes[0], palette='viridis')
    
# Set individual plot title and axis labels for make counts
axes[0].set_title('Total Count of Vehicles by Make (Make)')
axes[0].set_xlabel('Number of Vehicles')
axes[0].set_ylabel('Make')
    
# Annotate bars with total vehicle counts per make
for container in axes[0].containers:
    axes[0].bar_label(container, padding=3)

# Calculate the number of unique models available for each make
unique_models = dfs['uk_dataset'].groupby('make')['model'].nunique().sort_values(ascending=False)
    
# Generate horizontal bar plot to show variety of inventory
sns.barplot(x=unique_models.values, y=unique_models.index, ax=axes[1], palette='magma')
    
# Set individual plot title and labels for unique model counts
axes[1].set_title('Variety of Inventory: Number of Unique Models per Make')
axes[1].set_xlabel('Count of Unique Models')
axes[1].set_ylabel('Make')
    
# Annotate bars with the specific count of unique models
for container in axes[1].containers:
    axes[1].bar_label(container, padding=3)

# Optimize layout for readability and display the combined figure
plt.tight_layout()
# Show barplots
plt.show()